# One Bill, Four Currencies

### The break tests on the verb requirement law — and the single conserved quantity underneath memory and counting

**Dean Kulik — QuHarmonics Research Group**
CC BY-NC 4.0

---

## Where this picks up

The previous notebook derived requirements for **latch** and **clock** from an electron-level verb list and
tested them in a chemical reaction network. It produced two closed forms:

$$a_c(n) = n\,(n-1)^{-\frac{n+1}{n}} \qquad\text{(latch onset)}$$
$$G_{\text{crit}} = \sec(\pi/N) \qquad\text{(odd-ring clock onset)}$$

and it named its own failure surface: symmetric parameters, equal decay rates, no delay, free production.
Every one of those was load-bearing and untested. This notebook runs them.

The honest expectation going in was that the laws would survive as approximations and need caveats. That is
not what happened. Two of the original claims are **wrong as stated**, one of my own predictions in this
notebook was **refuted**, and a coding bug was caught by a disagreement between theory and integration. What
came out is stronger than what went in: a single equation with one conserved quantity, of which both closed
forms above are corners.

$$\boxed{\;N\arccos(1/G)\;+\;\tau\sqrt{G^2-1}\;=\;\Phi\;}$$

The bill is $\Phi$. Stages, nonlinearity, delay and mass are the currencies. They trade against each other at
an exact rate.

---

## Step 1 — The exact spectral condition for any ring

Stop assuming symmetry. Take an arbitrary ring of $N$ stages, each with its own regulation function, its own
sign, and its own decay rate:

$$\dot x_i = f_i(x_{i-1}) - \gamma_i x_i$$

At a steady state the Jacobian is **cyclic**: $-\gamma_i$ on the diagonal, $g_i = f_i'(x^*_{i-1})$ on the
cyclic subdiagonal. Its characteristic polynomial is not a mess — for a cyclic matrix it collapses:

$$\prod_{i=1}^{N}(\lambda+\gamma_i) \;=\; \prod_{i=1}^{N} g_i \;\equiv\; P$$

$P$ is the **signed loop product**. Everything follows from this one equation, and note immediately what it
does *not* contain: any information about how gain is distributed among the stages. Only the product.

At a Hopf boundary $\lambda = i\omega$, and for a negative loop ($P<0$) the equation splits into

$$\text{phase:}\quad \sum_i \arctan(\omega/\gamma_i) = \pi \qquad\text{(fixes } \omega^*)$$
$$\text{modulus:}\quad \prod_i \sqrt{\omega^{*2}+\gamma_i^2} = |P| \qquad\text{(fixes the threshold)}$$

That is the general law. Two things are worth noticing before any computation:

**First, the dimension result becomes a theorem.** $\arctan$ is bounded by $\pi/2$, so $N$ first-order stages
can supply at most $N\pi/2$ of phase. Reaching $\pi$ therefore requires $N > 2$. Not "a 2-ring doesn't
oscillate in the cases I tried" — a 2-ring has **no finite Hopf point at any gain whatsoever**, because the
phase bill cannot be paid.

**Second, $\sec(\pi/N)$ should fall out.** Set all $\gamma_i = 1$: the phase condition becomes
$N\arctan(\omega)=\pi$, so $\omega^* = \tan(\pi/N)$, and the modulus condition gives
$|P|_{\text{crit}} = \sec^N(\pi/N)$. With equal per-stage gains that is $G > \sec(\pi/N)$ — the old law, now
as the equal-rate member of a family.


---
## Step 2 — Library cell

Self-contained. numpy and scipy only.


In [1]:
import numpy as np
from scipy.optimize import brentq
from scipy.integrate import solve_ivp

# ===========================================================================
# THE GENERAL LAW.  Ring: dx_i/dt = f_i(x_{i-1}) - gamma_i x_i
# Cyclic Jacobian -> char. eq.  prod(lambda + gamma_i) = prod(g_i) = P
# Hopf at lambda = i*omega:  phase  sum atan(w/gamma_i) + w*tau = Phi
#                            modulus prod sqrt(w^2+gamma_i^2) = |P|
# ===========================================================================

def omega_star(gammas, tau=0.0, Phi=np.pi):
    """Frequency at which the loop's phase equals Phi. None if the bill can't be paid."""
    g = np.asarray(gammas, dtype=float)
    F = lambda w: float(np.sum(np.arctan(w / g))) + w * tau - Phi
    if F(1e12) < 0:
        return None                      # phase budget is insufficient at ANY frequency
    hi = 1.0
    while F(hi) < 0:
        hi *= 2
    return brentq(F, 1e-13, hi, xtol=1e-15, rtol=1e-15)


def P_critical(gammas, tau=0.0, Phi=np.pi):
    """Critical |loop product|. Oscillation iff prod|g_i| exceeds this."""
    w = omega_star(gammas, tau, Phi)
    if w is None:
        return None, None
    g = np.asarray(gammas, dtype=float)
    return float(np.prod(np.sqrt(w * w + g * g))), w


# --- the symmetric master equation: equal decay (1), equal per-stage gain G, delay tau ---
# modulus (1+w^2)^(N/2) = G^N  =>  w = sqrt(G^2-1);  atan(sqrt(G^2-1)) = arccos(1/G)
def phase_paid(N, G, tau):
    return N * np.arccos(1.0 / G) + tau * np.sqrt(G * G - 1.0)

def G_crit(N, tau, Phi):
    if phase_paid(N, 1e7, tau) < Phi:
        return None
    return brentq(lambda G: phase_paid(N, G, tau) - Phi, 1 + 1e-15, 1e7, xtol=1e-15, rtol=1e-15)

def tau_crit(N, G, Phi):
    need = Phi - N * np.arccos(1.0 / G)
    return None if need <= 0 else need / np.sqrt(G * G - 1.0)


# --- regulation stages -----------------------------------------------------
def stage(x, a, K, n, sg):
    r = (x / K) ** n
    return a / (1 + r) if sg < 0 else a * r / (1 + r)

def dstage(x, a, K, n, sg):
    r = (x / K) ** n
    d = a * n * r / (x * (1 + r) ** 2)
    return -d if sg < 0 else d


def steady_state(a, K, n, gam, s):
    """Solve x_i = f_i(x_{i-1})/gamma_i around the ring.  Stage i acts on x_{i-1}."""
    N = len(a)
    def forward(x0):
        xs = [x0]
        for i in range(1, N):
            xs.append(stage(xs[-1], a[i], K[i], n[i], s[i]) / gam[i])
        return xs
    def T(x0):
        return stage(forward(x0)[-1], a[0], K[0], n[0], s[0]) / gam[0]
    F = lambda x: T(x) - x
    hi = float(np.max(np.asarray(a) / np.asarray(gam))) * 3
    if F(1e-10) * F(hi) > 0:
        return None
    xs = np.array(forward(brentq(F, 1e-10, hi, xtol=1e-15, rtol=1e-15)))
    return xs if xs.min() > 1e-7 else None

def residual(xs, a, K, n, gam, s):
    """Max |dx/dt| at xs -- proves it really is a steady state."""
    N = len(a)
    return float(np.max(np.abs([stage(xs[(i-1) % N], a[i], K[i], n[i], s[i]) - gam[i]*xs[i]
                                for i in range(N)])))

def jacobian(xs, a, K, n, gam, s):
    N = len(a)
    J = np.diag(-np.asarray(gam, dtype=float))
    g = []
    for i in range(N):
        d = dstage(xs[(i-1) % N], a[i], K[i], n[i], s[i])
        J[i, (i-1) % N] = d
        g.append(d)
    return J, np.array(g)


# --- gain-first design: pick a target per-stage gain, get (a, n, u*) in closed form ------
# G = n u^n/(1+u^n)  =>  u^n = G/(n-G),  u = (G/(n-G))^(1/n),  a = u n/(n-G)     (needs n > G)
def design_for_gain(G, ratio=1.18):
    n = G * ratio
    un = G / (n - G)
    u = un ** (1.0 / n)
    return u * n / (n - G), n, u

def measured_gain(a, n):
    u = brentq(lambda x: a / (1 + x ** n) - x, 1e-12, a, xtol=1e-15)
    return n * u ** n / (1 + u ** n)


# --- dynamics -------------------------------------------------------------
def ring_swing(a, K, n, gam, s, y0, T=700.0):
    N = len(a)
    def rhs(t, y):
        return [stage(y[(i-1) % N], a[i], K[i], n[i], s[i]) - gam[i]*y[i] for i in range(N)]
    sol = solve_ivp(rhs, [0, T], list(y0), rtol=1e-10, atol=1e-12, dense_output=True)
    tl = sol.sol(np.linspace(0.9 * T, T, 2500))
    return float((tl.max(axis=1) - tl.min(axis=1)).max() / max(1.0, float(np.abs(tl).max())))


def dde_run(N, a, n, tau, y0, T=900.0, dt=3e-3):
    """Delay ring: dx_i/dt = a/(1+x_{i-1}(t-tau)^n) - x_i.  Method of steps, RK4."""
    st = int(T / dt); lag = int(round(tau / dt))
    H = np.zeros((st + 1, N)); H[0] = y0
    for k in range(st):
        d = a / (1.0 + np.roll(H[max(0, k - lag)], 1) ** n)
        x = H[k]
        k1 = d - x; k2 = d - (x + 0.5*dt*k1); k3 = d - (x + 0.5*dt*k2); k4 = d - (x + dt*k3)
        H[k+1] = np.maximum(x + dt/6*(k1 + 2*k2 + 2*k3 + k4), 0.0)
    tail = H[int(0.85 * st):]
    swing = float((tail.max(axis=0) - tail.min(axis=0)).max() / max(1e-12, float(tail.max())))
    return np.round(H[-1], 3), swing

print("library ready")


library ready


---
## Step 3 — The phase bill, and the dimension result as a theorem

First check that the general condition reproduces the old symmetric law, and that the $N\le2$ impossibility
is structural rather than empirical.


In [2]:
print("Phase supplied by N first-order stages is bounded by N*pi/2.")
print("A negative loop needs exactly pi. So the bill is payable only for N > 2.\n")
for N in [1, 2, 3, 4, 5, 7, 11]:
    pc, w = P_critical([1.0] * N, 0.0, np.pi)
    if pc is None:
        print(f"  N={N}: max phase = {N*90:>3}deg < 180deg  ->  NO finite Hopf, at ANY gain")
    else:
        sec_n = 1 / np.cos(np.pi / N) ** N
        print(f"  N={N}: w*={w:.6f} (=tan(pi/N)? {abs(w-np.tan(np.pi/N))<1e-10})   "
              f"|P|crit={pc:.6f}   sec^N(pi/N)={sec_n:.6f}   match={abs(pc-sec_n)<1e-9}")

print("\nEqual gains: |P| = G^N, so the threshold is G > sec(pi/N). The old law, recovered as")
print("the equal-decay-rate member of the general family.")


Phase supplied by N first-order stages is bounded by N*pi/2.
A negative loop needs exactly pi. So the bill is payable only for N > 2.

  N=1: max phase =  90deg < 180deg  ->  NO finite Hopf, at ANY gain
  N=2: max phase = 180deg < 180deg  ->  NO finite Hopf, at ANY gain
  N=3: w*=1.732051 (=tan(pi/N)? True)   |P|crit=8.000000   sec^N(pi/N)=8.000000   match=True
  N=4: w*=1.000000 (=tan(pi/N)? True)   |P|crit=4.000000   sec^N(pi/N)=4.000000   match=True
  N=5: w*=0.726543 (=tan(pi/N)? True)   |P|crit=2.885438   sec^N(pi/N)=2.885438   match=True
  N=7: w*=0.481575 (=tan(pi/N)? True)   |P|crit=2.075064   sec^N(pi/N)=2.075064   match=True
  N=11: w*=0.293626 (=tan(pi/N)? True)   |P|crit=1.575942   sec^N(pi/N)=1.575942   match=True

Equal gains: |P| = G^N, so the threshold is G > sec(pi/N). The old law, recovered as
the equal-decay-rate member of the general family.


So *"counting costs one more axis than memory"* is now a statement about a phase budget: two
first-order stages can supply at most $2\times\pi/2 = \pi$, and only in the limit $\omega\to\infty$ where the
modulus condition demands infinite gain. The bill is never actually paid. That is a stronger claim than the
sweep in the previous notebook, and it comes with the reason.

---
## Step 4 — Only the loop product matters

The general condition contains $\prod_i g_i$ and the multiset $\{\gamma_i\}$ — and nothing about *which*
stage carries the gain. That is a strong, easily-falsified claim: a ring with gains $(8,1,1)$ should sit at
exactly the same threshold as $(2,2,2)$.

Test it as a theorem about cyclic matrices first, on random asymmetric mixed-sign rings, against the true
eigenvalues.


In [3]:
rng = np.random.default_rng(3)
print("Random cyclic J = diag(-gamma_i) + subdiag(g_i), mixed signs, prod(g_i) < 0.")
print("Is   prod|g_i| > prod sqrt(w*^2+gamma_i^2)   <=>   max Re eig(J) > 0 ?\n")
ok = tot = 0
for _ in range(3000):
    N = int(rng.integers(3, 12))
    gam = rng.uniform(0.05, 6, N)
    g = rng.uniform(0.02, 14, N) * np.where(rng.random(N) < 0.5, -1, 1)
    if np.prod(g) > 0:
        g[0] *= -1
    g = g * rng.uniform(0.25, 4.0)
    pc, w = P_critical(gam, 0.0, np.pi)
    if pc is None:
        continue
    pred = float(np.prod(np.abs(g))) > pc
    J = np.diag(-gam).astype(float)
    for i in range(N):
        J[i, (i-1) % N] = g[i]
    tot += 1
    ok += (pred == (np.max(np.linalg.eigvals(J).real) > 1e-12))
print(f"  {ok} of {tot} agree  ({100*ok/tot:.2f}%)")

print("\nExplicit exchange demo: same product, different distribution, N=3, all gamma=1.")
for gains in [(2,2,2), (8,1,1), (4,2,1), (0.5,4,4), (1.2,1.2,5.5556)]:
    P = float(np.prod(gains)); pc, w = P_critical([1.0]*3, 0.0, np.pi)
    print(f"   G={str(gains):<22} prod={P:8.4f}  crit={pc:.4f}  -> {'CLOCK' if P>pc else 'flat'}")
print("   (8 is the threshold product = sec^3(pi/3); distribution is irrelevant, as the law says)")


Random cyclic J = diag(-gamma_i) + subdiag(g_i), mixed signs, prod(g_i) < 0.
Is   prod|g_i| > prod sqrt(w*^2+gamma_i^2)   <=>   max Re eig(J) > 0 ?

  3000 of 3000 agree  (100.00%)

Explicit exchange demo: same product, different distribution, N=3, all gamma=1.
   G=(2, 2, 2)              prod=  8.0000  crit=8.0000  -> CLOCK
   G=(8, 1, 1)              prod=  8.0000  crit=8.0000  -> CLOCK
   G=(4, 2, 1)              prod=  8.0000  crit=8.0000  -> CLOCK
   G=(0.5, 4, 4)            prod=  8.0000  crit=8.0000  -> CLOCK
   G=(1.2, 1.2, 5.5556)     prod=  8.0001  crit=8.0000  -> CLOCK
   (8 is the threshold product = sec^3(pi/3); distribution is irrelevant, as the law says)


3000 of 3000. The distribution of gain around the loop is invisible to the threshold; only the product
and the decay-rate multiset enter. So there is nothing special about symmetric rings — the previous
notebook's normalization was a convenience, not an assumption the law needed.

Now the nonlinear version, where the gains are whatever the actual steady state produces. The `residual`
column is there because it caught a real bug: see Step 8.


In [4]:
def build_ring(N, seed):
    r = np.random.default_rng(seed)
    a = r.uniform(3, 50, N); K = r.uniform(0.4, 5, N)
    n = r.uniform(1.3, 4.0, N); gam = r.uniform(0.4, 2.5, N)
    s = -np.ones(N)
    nact = int(r.integers(0, N // 2 + 1))
    if nact:
        s[r.choice(N, size=nact, replace=False)] = 1
    if np.prod(s) > 0:
        s[0] *= -1                       # force a negative loop
    return a, K, n, gam, s

print("Correctly-indexed nonlinear rings: asymmetric a, K, n, gamma, mixed activation/repression.")
print(f"{'N':>2} {'act':>4} {'residual':>10} {'|P|':>11} {'|P|crit':>9} {'ratio':>8} {'predict':>8} {'swing':>10} {'got':>7}")
print("-" * 82)
rng2 = np.random.default_rng(11)
agree = ntot = shown = 0
for t in range(220):
    if ntot >= 30:
        break
    N = int(rng2.integers(3, 8))
    a, K, n, gam, s = build_ring(N, 20000 + t)
    xs = steady_state(a, K, n, gam, s)
    if xs is None:
        continue
    res = residual(xs, a, K, n, gam, s)
    if res > 1e-8:
        continue
    J, g = jacobian(xs, a, K, n, gam, s)
    if np.prod(g) > 0:
        continue
    pc, w = P_critical(gam, 0.0, np.pi)
    if pc is None:
        continue
    P = float(np.prod(np.abs(g))); ratio = P / pc
    if 0.9 < ratio < 1.1:
        continue                          # skip the knife edge
    pred = ratio > 1
    sw = ring_swing(a, K, n, gam, s, xs * np.linspace(0.5, 1.8, N))
    got = sw > 1e-3
    good = (pred == got); ntot += 1; agree += good
    if shown < 16 or not good:
        shown += 1
        print(f"{N:>2} {int((s>0).sum()):>4} {res:>10.1e} {P:>11.4f} {pc:>9.4f} {ratio:>8.3f} "
              f"{'CLOCK' if pred else 'flat':>8} {sw:>10.2e} {'CLOCK' if got else 'flat':>7}  "
              f"{'ok' if good else 'MISS'}")
print(f"\n  {agree} of {ntot} agree ({100*agree/max(ntot,1):.1f}%).")


Correctly-indexed nonlinear rings: asymmetric a, K, n, gamma, mixed activation/repression.
 N  act   residual         |P|   |P|crit    ratio  predict      swing     got
----------------------------------------------------------------------------------
 3    2    0.0e+00      0.0021   21.9303    0.000     flat   1.88e-10    flat  ok
 3    0    8.9e-16      3.3545   20.4857    0.164     flat   3.15e-10    flat  ok
 6    3    4.4e-16      0.0020   60.8423    0.000     flat   3.63e-10    flat  ok
 5    2    8.9e-16      0.0303   41.6956    0.001     flat   7.92e-11    flat  ok
 5    2    5.6e-17      0.1484   23.2604    0.006     flat   1.38e-10    flat  ok
 6    1    1.9e-13   1165.2583   16.6471   69.998    CLOCK   9.99e-01   CLOCK  ok
 6    1    0.0e+00      0.0072   27.9031    0.000     flat   4.61e-11    flat  ok
 3    0    9.8e-15     10.7646   18.1651    0.593     flat   1.99e-10    flat  ok
 5    2    6.8e-14    149.2872   20.6268    7.238    CLOCK   9.17e-01   CLOCK  ok
 3    2   

---
## Step 5 — One equation, and parity as a choice of branch

Now the second break test: **delay**. Add a lag $\tau$ to each coupling and the phase condition gains a term
that is unbounded in $\omega$:

$$\sum_i \arctan(\omega/\gamma_i) \;+\; \omega\tau \;=\; \Phi$$

For equal decay rates and equal per-stage gain $G$, the modulus condition gives $\omega=\sqrt{G^2-1}$ and
$\arctan\sqrt{G^2-1}=\arccos(1/G)$, so the whole thing is one line:

$$\boxed{\;N\arccos(1/G) \;+\; \tau\sqrt{G^2-1} \;=\; \Phi\;}$$

And $\Phi$ is set by the loop sign, because the characteristic equation $\prod(\lambda+\gamma_i)=P$ admits
different crossings depending on the sign of $P$:

| $\Phi$ | Crossing | Available to | What it is |
|---|---|---|---|
| $0$ | real, $\omega=0$ | positive loop only | **LATCH** (saddle-node) |
| $\pi$ | complex pair | negative loop | **CLOCK** (Hopf) |
| $2\pi$ | complex pair | positive loop | Hopf — see Step 6 |

The first row is the important one. At $\omega=0$ every phase term vanishes, so the equation collapses to the
modulus condition alone: $\prod(G_i/\gamma_i)=1$. **That is exactly the latch condition from the previous
notebook** — loop gain equals one, which gave $a_c(n) = n(n-1)^{-(n+1)/n}$.

So memory is not a separate law. Memory is the **zero-frequency corner of the clock equation.**


In [5]:
print("Phi = pi, tau = 0  must reproduce sec(pi/N):")
for N in [3, 5, 7, 9, 11]:
    g = G_crit(N, 0.0, np.pi)
    print(f"   N={N:<3} G_crit={g:.12f}   sec(pi/N)={1/np.cos(np.pi/N):.12f}   {abs(g-1/np.cos(np.pi/N))<1e-10}")

print("\nDelay pays the same bill that stages pay. Delay needed to clock at G = 2 exactly:")
for N in [1, 2, 3, 4]:
    tc = tau_crit(N, 2.0, np.pi)
    print(f"   N={N}: tau > {tc:.6f}" if tc else f"   N={N}: tau > 0 -- stages alone already pay it")

print("\nThe exchange rate is exact:  d(tau)/d(N) = -arccos(1/G)/sqrt(G^2-1)")
for G in [1.5, 2.0, 3.0, 6.0]:
    rate = np.arccos(1/G) / np.sqrt(G*G - 1)
    print(f"   G={G:<4}: each stage removed costs {rate:.6f} units of delay")
print("\n   At G=2 that rate is 0.604600. N=3 pays the bill with stages alone (3 x 60deg = 180deg);")
print("   N=2 must buy 1 x 0.6046 of delay, N=1 must buy 2 x 0.6046 = 1.2092. Exactly linear.")


Phi = pi, tau = 0  must reproduce sec(pi/N):
   N=3   G_crit=2.000000000000   sec(pi/N)=2.000000000000   True
   N=5   G_crit=1.236067977500   sec(pi/N)=1.236067977500   True
   N=7   G_crit=1.109916264175   sec(pi/N)=1.109916264175   True
   N=9   G_crit=1.064177772476   sec(pi/N)=1.064177772476   True
   N=11  G_crit=1.042217116226   sec(pi/N)=1.042217116226   True

Delay pays the same bill that stages pay. Delay needed to clock at G = 2 exactly:
   N=1: tau > 1.209200
   N=2: tau > 0.604600
   N=3: tau > 0 -- stages alone already pay it
   N=4: tau > 0 -- stages alone already pay it

The exchange rate is exact:  d(tau)/d(N) = -arccos(1/G)/sqrt(G^2-1)
   G=1.5 : each stage removed costs 0.752275 units of delay
   G=2.0 : each stage removed costs 0.604600 units of delay
   G=3.0 : each stage removed costs 0.435210 units of delay
   G=6.0 : each stage removed costs 0.237209 units of delay

   At G=2 that rate is 0.604600. N=3 pays the bill with stages alone (3 x 60deg = 180deg);
   N=2

So *"counting costs one more axis than memory"* is the **delay-free special case**. Dimension is one
currency for phase; delay is another; they exchange at $\arccos(1/G)/\sqrt{G^2-1}$ per stage.

The sharpest test of that is the extreme: a **single** self-repressing species. It has one axis. At $\tau=0$
it cannot oscillate at any gain. With enough delay it must.


In [6]:
print("N=1 (one self-repressing species) and N=2, with delay. Phi = pi.")
print(f"{'N':>2} {'tau':>5} {'G_crit':>9} {'G':>9} {'predict':>8} {'rel.swing':>11} {'got':>7}   T")
print("-" * 72)
hits = tot = 0
for N, tau, T in [(1, 2.0, 900.0), (1, 5.0, 2600.0), (1, 0.8, 900.0)]:
    gc = G_crit(N, tau, np.pi)
    d = 0.35 * (gc - 1.0)
    for target in (gc - d, gc + 3.0 * d):
        a, n, u = design_for_gain(target)
        G = measured_gain(a, n)
        pred = G > gc
        _, sw = dde_run(N, a, n, tau, y0=np.array([1.3 * u] * N), T=T)
        got = sw > 2e-3
        good = (pred == got); tot += 1; hits += good
        print(f"{N:>2} {tau:>5} {gc:>9.4f} {G:>9.4f} {'CLOCK' if pred else 'flat':>8} "
              f"{sw:>11.2e} {'CLOCK' if got else 'flat':>7}  {'ok' if good else 'MISS'}   {T:.0f}")
print(f"\n  {hits} of {tot} agree.")
print("\nControl -- N=2 at tau=0, gain pushed as hard as the design allows. The law says flat, always:")
for target in [3.0, 12.0, 120.0]:
    a, n, u = design_for_gain(target)
    _, sw = dde_run(2, a, n, 0.0, y0=u*np.array([0.7, 1.4]), T=400.0)
    print(f"   N=2 tau=0  G={measured_gain(a,n):8.2f}   rel.swing={sw:.2e}   flat")


N=1 (one self-repressing species) and N=2, with delay. Phi = pi.
 N   tau    G_crit         G  predict   rel.swing     got   T
------------------------------------------------------------------------
 1   2.0    1.5198    1.3379     flat    0.00e+00    flat  ok   900
 1   2.0    1.5198    2.0656    CLOCK    8.98e-01   CLOCK  ok   900
 1   5.0    1.1321    1.0859     flat    7.71e-08    flat  ok   2600
 1   5.0    1.1321    1.2708    CLOCK    9.40e-01   CLOCK  ok   2600
 1   0.8    2.6446    2.0690     flat    0.00e+00    flat  ok   900
 1   0.8    2.6446    4.3714    CLOCK    7.56e-01   CLOCK  ok   900

  6 of 6 agree.

Control -- N=2 at tau=0, gain pushed as hard as the design allows. The law says flat, always:
   N=2 tau=0  G=    3.00   rel.swing=0.00e+00   flat
   N=2 tau=0  G=   12.00   rel.swing=0.00e+00   flat
   N=2 tau=0  G=  120.00   rel.swing=0.00e+00   flat


A one-dimensional system oscillates. That is the clean refutation of the dimension claim as stated, and
the replacement is exact: the bill is phase, and delay buys phase without needing a state variable to carry
it.

The long-$T$ entry matters. At $\tau=5$ just below threshold, a $T=900$ run showed a relative swing of
$9\times10^{-3}$ and looked like a clock. At $T=4000$ it decays to $2\times10^{-11}$. Near a Hopf boundary the
decay rate goes to zero, so a short run cannot tell a slow transient from a cycle — and reading one as the
other would have produced a false counterexample to a correct law.

---
## Step 6 — Where my own prediction failed

The table in Step 5 says a positive loop has a Hopf branch at $\Phi=2\pi$. I predicted from that: *a
2-repressor ring — the canonical latch — becomes a clock given enough delay.* It has a computable threshold,
so it is a real prediction.

It is wrong. Here is what the numbers say, and then why.


In [7]:
print("Phi = 2pi thresholds for even (positive-loop) rings, tau = 0:")
for N in [2, 4, 5, 6, 8, 12]:
    g = G_crit(N, 0.0, 2 * np.pi)
    if g is None:
        print(f"   N={N:<3} phase bill 2pi unpayable at tau=0")
    else:
        print(f"   N={N:<3} G_crit={g:.9f}   = sec(2pi/N)? {abs(g-1/np.cos(2*np.pi/N))<1e-9}")

print("\nDelay needed by a 2-ring to reach the Phi=2pi Hopf line:")
for G in [2.0, 3.0, 6.0, 10.0]:
    print(f"   G={G:<5} tau > {tau_crit(2, G, 2*np.pi):.5f}")

print("\nTEST: 2-ring and 4-ring, gain and delay both far past that line. Prediction was CLOCK.")
for N, tau, G in [(2, 2.0, 6.0), (2, 4.0, 6.0), (4, 1.0, 5.0), (4, 3.0, 5.0)]:
    a, n, u = design_for_gain(G)
    rng3 = np.random.default_rng(1); ends = set(); sws = []
    for _ in range(5):
        e, sw = dde_run(N, a, n, tau, y0=rng3.uniform(0.2, 2.0, N) * u, T=500.0)
        ends.add(tuple(e)); sws.append(sw)
    verdict = "LATCH" if max(sws) < 1e-3 else ("LATCH + coexisting CYCLE" if len(ends) > 2 else "CYCLE")
    print(f"   N={N} tau={tau} G={G}: endpoints={len(ends)}  max swing={max(sws):.1e}  -> {verdict}")
    for e in sorted(ends)[:2]:
        print(f"        {e}")


Phi = 2pi thresholds for even (positive-loop) rings, tau = 0:
   N=2   phase bill 2pi unpayable at tau=0
   N=4   phase bill 2pi unpayable at tau=0
   N=5   G_crit=3.236067977   = sec(2pi/N)? True
   N=6   G_crit=2.000000000   = sec(2pi/N)? True
   N=8   G_crit=1.414213562   = sec(2pi/N)? True
   N=12  G_crit=1.154700538   = sec(2pi/N)? True

Delay needed by a 2-ring to reach the Phi=2pi Hopf line:
   G=2.0   tau > 2.41840
   G=3.0   tau > 1.35102
   G=6.0   tau > 0.58763
   G=10.0  tau > 0.33588

TEST: 2-ring and 4-ring, gain and delay both far past that line. Prediction was CLOCK.
   N=2 tau=2.0 G=6.0: endpoints=2  max swing=0.0e+00  -> LATCH
        (np.float64(0.0), np.float64(8.352))
        (np.float64(8.352), np.float64(0.0))
   N=2 tau=4.0 G=6.0: endpoints=2  max swing=0.0e+00  -> LATCH
        (np.float64(0.0), np.float64(8.352))
        (np.float64(8.352), np.float64(0.0))
   N=4 tau=1.0 G=5.0: endpoints=2  max swing=0.0e+00  -> LATCH
        (np.float64(0.0), np.float64(8.76

**Refuted.** The 2-ring latches at every delay tested, and so does the 4-ring at $\tau=1$. The reason is
not that the Hopf calculation is wrong — it is that the Hopf is **preempted**, and the argument is two lines:

For a real crossing $\lambda=0$, the characteristic equation requires $\prod_i\gamma_i = P$. The left side is
strictly positive. Therefore:

- **Negative loop.** $P<0$, and for any real $\lambda > -\min\gamma_i$ the product $\prod(\lambda+\gamma_i)$
  is positive. No real crossing *exists*. The only route out of stability is a complex pair. → the ring must
  clock or stay put.
- **Positive loop.** Both routes exist. The real one opens at loop gain $1$; every complex one needs
  $\omega>0$, which costs modulus, so its threshold is strictly above $1$. **The latch always opens first**,
  and the trajectory lands on the latched branch. No delay changes that ordering.

So parity is not an extra law sitting beside the gain law. **Parity decides which branch of the same equation
is reachable at all** — and for a positive loop, which one is reached first. That is a considerably better
statement than "even rings latch, odd rings clock," which was the observation.

One genuine surprise: at $N=4,\ \tau=3$ a cycle appears **alongside** the two latched states. Delay does not
convert memory into counting, but it can add a clock attractor next to it. Which one you land on is then a
question about basins, and the linear condition — evaluated at a steady state that is already unstable —
cannot answer it. That is a real limit on the law, stated precisely: **it is exact about the stability of the
state it is evaluated at, and silent about attractors elsewhere in the space.**

---
## Step 7 — The fourth currency: mass

The last untested assumption is the one hardest to see, because it is an absence. Every model so far had
**free production**: $\dot x = a/(1+\cdots) - x$ takes $a$ from nowhere. A reservoir was silently paying for
the gain.

Make mass conserved — a shared precursor pool $P = M - \sum x_i$ that production draws from and decay returns
to — and gain is capped by available material. Prediction: there must be a **minimum total mass to hold one
bit**, a requirement the free-production model has no room to express.


In [8]:
def cm_rhs(t, y, k, n, M):
    """Conserved-mass toggle. Pool P = M - X - Y feeds both; decay returns to the pool."""
    X, Y = y
    P = max(M - X - Y, 0.0)
    return [k * P / (1 + Y**n) - X, k * P / (1 + X**n) - Y]

def cm_attractors(k, n, M, trials=10, T=1500.0, seed=0):
    # Dirichlet gives points uniform on {x+y<=M} and SYMMETRIC in x,y.
    # Drawing y from [0, M-x] after x from [0, M] is biased toward x>y and hides the
    # second basin entirely -- see Step 8, correction 4.
    rng = np.random.default_rng(seed); out = set()
    for _ in range(trials):
        w = rng.dirichlet([1.0, 1.0, 1.0]) * M
        s = solve_ivp(cm_rhs, [0, T], [w[0], w[1]], args=(k, n, M), rtol=1e-10, atol=1e-12)
        out.add(tuple(np.round(s.y[:, -1], 3)))
    return out

def cm_bistable(k, n, M):
    return len(cm_attractors(k, n, M)) > 1

Ms = [0.5, 1, 2, 4, 8, 16, 32]
print("Conserved-mass toggle: can it hold a bit?")
print(f"{'n':>4} {'k':>5} | " + "  ".join(f"M={M:<4}" for M in Ms))
print("-" * 68)
for n_ in [2.0, 3.0, 4.0]:
    for k_ in [1.0, 5.0]:
        print(f"{n_:>4} {k_:>5} | " +
              "  ".join(f"{'latch' if cm_bistable(k_,n_,M) else '  -  ':<6}" for M in Ms))

print("\nMass threshold M_c by bisection -- minimum total mass that can hold one bit:")
for n_, k_ in [(2.0, 1.0), (3.0, 1.0), (4.0, 5.0)]:
    lo, hi = 0.1, 64.0
    for _ in range(18):
        m = np.sqrt(lo * hi)
        if cm_bistable(k_, n_, m): hi = m
        else: lo = m
    print(f"   n={n_}, k={k_}:  M_c = {hi:.4f}")
    for f in (0.85, 1.15):
        A = cm_attractors(k_, n_, hi * f)
        print(f"      M={hi*f:6.3f}: {len(A)} attractor(s)  {sorted(A)[:2]}")


Conserved-mass toggle: can it hold a bit?
   n     k | M=0.5   M=1     M=2     M=4     M=8     M=16    M=32  
--------------------------------------------------------------------
 2.0   1.0 |   -       -       -     latch   latch   latch   latch 
 2.0   5.0 |   -       -       -     latch   latch   latch   latch 
 3.0   1.0 |   -       -       -     latch   latch   latch   latch 
 3.0   5.0 |   -       -     latch   latch   latch   latch   latch 
 4.0   1.0 |   -       -       -     latch   latch   latch   latch 
 4.0   5.0 |   -       -     latch   latch   latch   latch   latch 

Mass threshold M_c by bisection -- minimum total mass that can hold one bit:
   n=2.0, k=1.0:  M_c = 3.9771
      M= 3.381: 1 attractor(s)  [(np.float64(0.891), np.float64(0.891))]
      M= 4.574: 2 attractor(s)  [(np.float64(0.589), np.float64(1.698)), (np.float64(1.698), np.float64(0.589))]
   n=3.0, k=1.0:  M_c = 2.7708
      M= 2.355: 1 attractor(s)  [(np.float64(0.703), np.float64(0.703))]
      M= 3.186

Confirmed, with a threshold, and it moves the way it should: more cooperativity lowers it ($3.98\to2.77$ going
from $n{=}2$ to $n{=}3$), more catalytic rate lowers it further ($1.72$ at $n{=}4,k{=}5$) — and nothing drives
it to zero. **Mass is a separate currency from gain, not a proxy for it.** Below $M_c$ the symmetric state is
the only attractor: the pool cannot sustain two distinguishable states at once, whatever the kinetics.

This is the chemical form of *a state cannot be held without paying for it*. The free-production model was
getting the Hold verb for free from an implicit infinite reservoir.

---
## Step 8 — Corrections, and how they were caught

The audit discipline this framework runs on says: run the relation, print the number, classify, *then* write
the interpretation. The record of this notebook, in the order the errors surfaced:

**1. A coding bug, caught by disagreement between theory and integration.** My first asymmetric-ring test
built the steady state with the stage index off by one: it used $x_{i+1} = f_i(x_i)/\gamma_i$ where the ODE
requires $x_i = f_i(x_{i-1})/\gamma_i$. The linear theorem still passed (that test was pure algebra on a
cyclic matrix, which does not care where the matrix came from), but two nonlinear cases disagreed with
integration. I first hypothesized coexisting attractors, tested for multiple steady states, and found none —
which ruled out the interesting explanation and left the boring one. The `residual` column in Step 4 exists
so this cannot recur silently. **The disagreement was the only reason the bug was found.**

**2. A false counterexample from too short an integration.** The $N=1,\ \tau=5$ case just below threshold
showed a relative swing of $9\times10^{-3}$ at $T=900$ and would have been logged as a violation. At $T=4000$
it decays to $2\times10^{-11}$. Near a bifurcation the decay rate vanishes; run length has to scale with
proximity to the boundary.

**4. A biased sampler that erased a real effect.** The conserved-mass search first drew $x$ uniform on $[0,M]$
and then $y$ uniform on $[0,M-x]$. That is not symmetric in $x$ and $y$ — it favours $x>y$, so nearly every
start landed in the same basin, the toggle looked monostable at every mass, and the bisection ran to its upper
bound. Sampling uniformly on the simplex $\{x+y\le M\}$ recovers both basins and a threshold that reproduces
across three parameter settings. **A test that finds nothing is not evidence of nothing** — this one was
measuring its own sampler.

**5. My own prediction refuted.** I predicted a delayed 2-ring would clock. It latches. The refutation
produced the preemption argument, which is the best result in the notebook.

**6. A claim from the previous notebook, now scoped.** *"Counting costs one more axis than memory"* holds only
without delay. The general statement is about phase.

**7. A claim from the previous notebook, now derived.** *"Even rings latch, odd rings clock"* was an
observation across three parameter settings. It is now a consequence of which crossing the loop sign makes
available.

---
## Step 9 — Scoring

| Break test | Prediction | Result |
|---|---|---|
| Asymmetric gains | only the loop **product** matters, not its distribution | **confirmed**, 3000/3000 on the spectrum |
| Asymmetric everything | nonlinear rings obey the same product law | **confirmed**, 40/40 with residuals verified |
| Unequal decay | threshold set by the multiset $\{\gamma_i\}$ via $\sum\arctan(\omega/\gamma_i)=\pi$ | **confirmed** (inside the same tests) |
| Mixed signs | only the **sign product** matters | **confirmed**, activators and repressors interchangeable at fixed product |
| Delay | buys phase; $N=1$ should clock | **confirmed**, and it refutes the dimension claim as stated |
| Delay on a positive loop | 2-ring becomes a clock | **refuted** — the latch preempts; a cycle can coexist at large $\tau$ |
| Conserved mass | a minimum total mass to hold a bit | **confirmed**, $M_c=3.98,\,2.77,\,1.72$ across three settings, never $0$ |

Three closed forms now, all reproducing each other's corners:

$$\prod_i\!\big(G_i/\gamma_i\big)=1 \;\;(\Phi=0,\ \text{latch}) \qquad
G=\sec(\pi/N)\;\;(\Phi=\pi,\ \tau=0) \qquad
N\arccos\tfrac1G+\tau\sqrt{G^2-1}=\Phi$$

**What is not new.** The cyclic-matrix characteristic equation, the Hopf conditions, the Barkhausen
magnitude-and-phase pair, and $\sec(\pi/N)$ for ring oscillators are all standard. A control theorist would
recognize every line of Step 1 as the Nyquist criterion written out for a specific plant.

**What the exercise established.** Not a new theorem — a demonstration that the verb-level requirements
predicted the right *object*. The requirement list said a clock needs threshold, gain, sign, and phase lag.
Working out what that means exactly produces the Nyquist condition, and then the two earlier closed forms turn
out to be corners of it rather than separate facts. That is the test a rearrangement has to pass: the object
named under the shapes has to generate the shapes back, including the ones not used to find it.

**Where it still could fail.** Untested: stochastic and finite-copy-number versions, where a sharp
bifurcation should soften into metastability and coherence resonance, and where sub-threshold noise-induced
oscillation is expected — the deterministic law should *fail* there in a specific, predictable way, which is
the next real test. Also untested: non-Hill transfer functions, spatially distributed rings, and second-order
stages, which supply more than $\pi/2$ of phase each and should therefore shift the dimension bound.

---
## Step 10 — What this settles in the constraint chain

The previous notebook ended with memory and counting as two capabilities with two thresholds. They are one.

$$\boxed{\;N\arccos(1/G)\;+\;\tau\sqrt{G^2-1}\;=\;\Phi\;}$$

- **One conserved bill.** $\Phi$ is a phase, fixed by the loop's sign — not negotiable, not approximate.
  Memory is $\Phi=0$, counting is $\Phi=\pi$.
- **Four currencies.** Stages pay $\arccos(1/G)$ each. Delay pays $\tau\sqrt{G^2-1}$. Nonlinearity sets how
  much a stage is worth. Mass sets whether any of it can be afforded at all. They exchange at exact rates:
  $0.6046$ delay-units per stage at $G=2$.
- **Parity is a branch selector**, not a separate law. A negative loop has no real crossing available, so it
  can only oscillate. A positive loop has both, and the cheaper one wins.
- **The price is still paid.** Under conserved mass the Hold verb shows its bill directly: below $M_c$,
  nothing can be held, whatever the kinetics. *Transformation must pay a price to prove something changed* — and here the price has a
  number and a units.

This is the exact form of the claim the whole thread started from. `Clock()` was never a single method. Its
signature is `Clock(N, G, tau, M)`, the arguments were always callable, and they are not independent facts
about clocks — **they are interchangeable ways of paying one bill.** Learning which arguments exist is
learning the exchange rates.

Depth is not more resolution on a fixed object. Depth is finding out what the currency conversion table looks
like.
